# OpenAI embeddings — service test

Tests `OpenAIEmbeddingsClient`: single embed, batch embed, similarity sanity-check.

**Prereqs:** `OPENAI_API_KEY` in `.env`. Costs ~$0.0001 total.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.services.embeddings.factory import make_openai_embeddings_client

embedder = make_openai_embeddings_client()
embedder.settings.model, embedder.settings.dimensions

In [ ]:
# Single text → one 1024-dim vector
vec = await embedder.embed_text('multi-head attention in transformers')
print('dims:', len(vec))
print('first 5:', [round(x, 4) for x in vec[:5]])

In [ ]:
# Batch: 3 texts → 3 vectors, order preserved
texts = [
    'attention mechanisms in neural networks',   # similar to query above
    'BERT pre-training objectives',              # related field
    'chocolate chip cookie recipe',              # unrelated
]
vecs = await embedder.embed_batch(texts)
print(len(vecs), 'vectors of', len(vecs[0]), 'dims')

In [ ]:
# Sanity: cosine similarity should rank attention > BERT > cookies
import math

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na  = math.sqrt(sum(x*x for x in a))
    nb  = math.sqrt(sum(x*x for x in b))
    return dot / (na * nb)

for text, v in zip(texts, vecs):
    print(f'{cosine(vec, v):.3f}  {text}')